# Assignment 1: Titanic Dataset Analysis
## Student: Berny Aopare

### Objective
This notebook performs data analysis and machine learning on the Titanic dataset to predict passenger survival.

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Set visualization style
sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Load the training dataset
df = pd.read_csv('train.csv')
print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Data Analysis (Minimum 3 Required)

### Analysis 1: Data Summary and Descriptive Statistics

In [ ]:
# Display basic information about the dataset
print("=" * 50)
print("DATASET INFORMATION")
print("=" * 50)
df.info()
print("\n" + "=" * 50)
print("MISSING VALUES")
print("=" * 50)
print(df.isnull().sum())
print("\n" + "=" * 50)
print("DESCRIPTIVE STATISTICS")
print("=" * 50)
df.describe()

In [ ]:
# Calculate standard deviation for numerical columns
print("=" * 50)
print("STANDARD DEVIATION")
print("=" * 50)
numerical_cols = ['Age', 'Fare', 'SibSp', 'Parch']
for col in numerical_cols:
    std = df[col].std()
    mean = df[col].mean()
    print(f"{col}: Mean = {mean:.2f}, Std Dev = {std:.2f}")

### Analysis 2: Survival Rate Analysis and Visualization

In [ ]:
# Survival rate overall
survival_rate = df['Survived'].value_counts(normalize=True) * 100
print("Overall Survival Rate:")
print(f"Died: {survival_rate[0]:.2f}%")
print(f"Survived: {survival_rate[1]:.2f}%")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Survival count
df['Survived'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['#d62728', '#2ca02c'])
axes[0, 0].set_title('Survival Count', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Survived (0=No, 1=Yes)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_xticklabels(['Died', 'Survived'], rotation=0)

# Survival by Gender
pd.crosstab(df['Sex'], df['Survived']).plot(kind='bar', ax=axes[0, 1], color=['#d62728', '#2ca02c'])
axes[0, 1].set_title('Survival by Gender', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Gender')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend(['Died', 'Survived'])
axes[0, 1].set_xticklabels(['Female', 'Male'], rotation=0)

# Survival by Class
pd.crosstab(df['Pclass'], df['Survived']).plot(kind='bar', ax=axes[1, 0], color=['#d62728', '#2ca02c'])
axes[1, 0].set_title('Survival by Passenger Class', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Passenger Class')
axes[1, 0].set_ylabel('Count')
axes[1, 0].legend(['Died', 'Survived'])
axes[1, 0].set_xticklabels(['1st Class', '2nd Class', '3rd Class'], rotation=0)

# Age distribution by survival
df[df['Survived']==0]['Age'].dropna().hist(ax=axes[1, 1], bins=30, alpha=0.5, label='Died', color='#d62728')
df[df['Survived']==1]['Age'].dropna().hist(ax=axes[1, 1], bins=30, alpha=0.5, label='Survived', color='#2ca02c')
axes[1, 1].set_title('Age Distribution by Survival', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Age')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### Analysis 3: Histogram Analysis of Key Features

In [ ]:
# Create histograms for numerical features
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Age histogram
df['Age'].hist(bins=30, ax=axes[0, 0], edgecolor='black', color='skyblue')
axes[0, 0].set_title('Age Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['Age'].mean(), color='red', linestyle='--', label=f'Mean: {df["Age"].mean():.1f}')
axes[0, 0].legend()

# Fare histogram
df['Fare'].hist(bins=30, ax=axes[0, 1], edgecolor='black', color='lightcoral')
axes[0, 1].set_title('Fare Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Fare')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(df['Fare'].mean(), color='red', linestyle='--', label=f'Mean: {df["Fare"].mean():.1f}')
axes[0, 1].legend()

# SibSp histogram
df['SibSp'].hist(bins=8, ax=axes[1, 0], edgecolor='black', color='lightgreen')
axes[1, 0].set_title('Siblings/Spouses Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Number of Siblings/Spouses')
axes[1, 0].set_ylabel('Frequency')

# Parch histogram
df['Parch'].hist(bins=7, ax=axes[1, 1], edgecolor='black', color='plum')
axes[1, 1].set_title('Parents/Children Distribution', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Number of Parents/Children')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### Analysis 4: Correlation Heatmap (Bonus)

In [ ]:
# Calculate correlation matrix
correlation_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
correlation_matrix = df[correlation_cols].corr()

# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Correlations with Survival:")
print(correlation_matrix['Survived'].sort_values(ascending=False))

## 3. Data Preprocessing for Model Training

In [ ]:
# Create a copy for preprocessing
df_model = df.copy()

# Fill missing values
df_model['Age'].fillna(df_model['Age'].median(), inplace=True)
df_model['Embarked'].fillna(df_model['Embarked'].mode()[0], inplace=True)
df_model['Fare'].fillna(df_model['Fare'].median(), inplace=True)

# Encode categorical variables
le_sex = LabelEncoder()
df_model['Sex'] = le_sex.fit_transform(df_model['Sex'])

le_embarked = LabelEncoder()
df_model['Embarked'] = le_embarked.fit_transform(df_model['Embarked'])

# Create family size feature
df_model['FamilySize'] = df_model['SibSp'] + df_model['Parch'] + 1

# Select features for training
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize']
X = df_model[features]
y = df_model['Survived']

print("Data preprocessing completed!")
print(f"Features used: {features}")
print(f"Training data shape: {X.shape}")

## 4. Train Machine Learning Model

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
model.fit(X_train, y_train)

print("Model training completed!")

## 5. Model Accuracy and Evaluation

In [ ]:
# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print("=" * 60)
print("MODEL ACCURACY")
print("=" * 60)
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"Testing Accuracy: {test_accuracy * 100:.2f}%")
print("=" * 60)

In [ ]:
# Detailed classification report
print("\nCLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred_test, target_names=['Died', 'Survived']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Died', 'Survived'], 
            yticklabels=['Died', 'Survived'])
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance in Predicting Survival', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nFeature Importance:")
print(feature_importance)

## 6. Summary

### Key Findings:

1. **Data Analysis Results:**
   - The dataset contains 891 passengers with 12 features
   - Overall survival rate is approximately 38%
   - Females had significantly higher survival rates than males
   - First-class passengers had better survival rates than other classes
   - Age and fare distributions show interesting patterns related to survival

2. **Model Performance:**
   - Algorithm: Random Forest Classifier
   - The model achieved good accuracy on both training and testing data
   - Most important features: Sex, Fare, Age, and Passenger Class

3. **Problem Classification:**
   - **Supervised Learning** (we have labeled data - Survived column)
   - **Binary Classification** (predicting two classes: survived or died)
   - **Batch Learning** (training on a fixed dataset)